# 🧩 Mini-Lab: The Auto-Grader

**Module 9: LLM Evaluation** | **Type: Mini-Lab (Brick)**

---

## Learning Objectives

By the end of this mini-lab, you will be able to:

1. **Understand** why generic evaluation criteria fail for domain-specific tasks
2. **Design** custom rubrics with weighted criteria tailored to specific use cases
3. **Implement** an auto-grader that uses an LLM judge with a rubric to produce structured, reproducible scores
4. **Compare** how the same response scores differently under different rubrics

## Target Concepts

| Concept | Description |
|---------|-------------|
| Rubric Scoring | Custom evaluation criteria with defined score levels and weights — enabling domain-specific, consistent quality assessment |
| LLM-as-Judge | Using an LLM to evaluate outputs against a rubric — the engine that powers the auto-grader |

## From Generic Judging to Custom Rubrics

In `mini-llm-judge`, we asked an LLM to score responses on generic dimensions like helpfulness, specificity, and clarity. That works for general-purpose evaluation, but **domain-specific tasks need domain-specific criteria**.

Consider these tasks:

| Task | What "Good" Means |
|------|-------------------|
| Medical patient summary | Accurate, includes key findings, no hallucinated conditions |
| Customer support reply | Empathetic tone, addresses the issue, offers a resolution |
| Code documentation | Covers parameters, return values, includes an example |

A **rubric** formalizes what "good" means by defining:
- **Criteria** — the specific dimensions to evaluate
- **Score levels** — what each score (1–5) means for that criterion
- **Weights** — how much each criterion matters relative to others

## Setup

In [2]:
import os
import re
from dotenv import load_dotenv
import openai
from IPython.display import display, Markdown

load_dotenv()

client = openai.OpenAI()

def md(text):
    """Display text as rendered markdown."""
    display(Markdown(text))

## Defining a Custom Rubric

Let's build a rubric for evaluating **customer support replies**. Each criterion has a name, description, weight, and score-level definitions so the judge knows exactly what each score means.

In [3]:
support_rubric = {
    "name": "Customer Support Reply",
    "criteria": [
        {
            "name": "Empathy",
            "description": "Does the reply acknowledge the customer's frustration and show understanding?",
            "weight": 0.3,
            "levels": {
                1: "No acknowledgment of the customer's feelings",
                2: "Minimal acknowledgment, feels robotic",
                3: "Acknowledges the issue but lacks warmth",
                4: "Warm and understanding tone",
                5: "Genuinely empathetic, makes the customer feel heard"
            }
        },
        {
            "name": "Resolution",
            "description": "Does the reply offer a clear, actionable solution to the problem?",
            "weight": 0.5,
            "levels": {
                1: "No solution offered",
                2: "Vague suggestion with no clear action",
                3: "Offers a solution but missing key details",
                4: "Clear solution with actionable steps",
                5: "Complete solution with steps, timeline, and follow-up"
            }
        },
        {
            "name": "Professionalism",
            "description": "Is the reply professional, well-structured, and free of errors?",
            "weight": 0.2,
            "levels": {
                1: "Unprofessional or rude tone",
                2: "Casual with grammatical errors",
                3: "Acceptable but could be more polished",
                4: "Professional and well-written",
                5: "Impeccable — polished, clear, and brand-appropriate"
            }
        }
    ]
}

print(f"Rubric: {support_rubric['name']}")
print(f"Criteria: {len(support_rubric['criteria'])}")
print(f"Weights sum: {sum(c['weight'] for c in support_rubric['criteria'])}")

Rubric: Customer Support Reply
Criteria: 3
Weights sum: 1.0


## Building the Rubric-Based Judge Prompt

The key to a good auto-grader is converting the rubric into a clear, structured prompt. The judge needs to see the score-level definitions so it understands what each number means.

In [4]:
def build_rubric_prompt(rubric, task_description, response):
    """Convert a rubric dict into a judge prompt."""
    criteria_text = ""
    for c in rubric["criteria"]:
        levels_text = "\n".join(f"      {k}: {v}" for k, v in c["levels"].items())
        criteria_text += (
            f"\n  - **{c['name']}** (weight: {c['weight']})\n"
            f"    {c['description']}\n"
            f"    Score levels:\n{levels_text}\n"
        )

    # Build the expected output format line
    format_lines = "\n".join(
        f"{c['name']}: <score>/5 - <justification>" for c in rubric["criteria"]
    )

    prompt = f"""You are an expert evaluator using a custom rubric.

**Task:** {task_description}

**Response to evaluate:**
{response}

**Rubric: {rubric['name']}**
{criteria_text}
Score the response on EACH criterion. Use the score levels above to guide your rating.

Format your answer EXACTLY like this:
{format_lines}
"""
    return prompt

# Preview the prompt structure
sample_prompt = build_rubric_prompt(
    support_rubric,
    "Reply to a customer who received a damaged product.",
    "(sample response)"
)
print(sample_prompt[:600] + "\n...")

You are an expert evaluator using a custom rubric.

**Task:** Reply to a customer who received a damaged product.

**Response to evaluate:**
(sample response)

**Rubric: Customer Support Reply**

  - **Empathy** (weight: 0.3)
    Does the reply acknowledge the customer's frustration and show understanding?
    Score levels:
      1: No acknowledgment of the customer's feelings
      2: Minimal acknowledgment, feels robotic
      3: Acknowledges the issue but lacks warmth
      4: Warm and understanding tone
      5: Genuinely empathetic, makes the customer feel heard

  - **Resolution** (weigh
...


## The Auto-Grader Function

Now we combine everything: the rubric prompt builder, the LLM judge call, score parsing, and **weighted scoring**.

In [5]:
def auto_grade(rubric, task_description, response, model="gpt-4o-mini"):
    """Grade a response using a rubric-based LLM judge."""
    prompt = build_rubric_prompt(rubric, task_description, response)
    
    result = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    judgment = result.choices[0].message.content
    
    # Parse scores
    scores = {}
    for c in rubric["criteria"]:
        match = re.search(rf"{c['name']}:\s*(\d)/5\s*-\s*(.+)", judgment)
        if match:
            scores[c["name"]] = {
                "score": int(match.group(1)),
                "justification": match.group(2).strip(),
                "weight": c["weight"]
            }
    
    # Compute weighted total
    weighted_score = sum(
        s["score"] * s["weight"] for s in scores.values()
    )
    
    return {"scores": scores, "weighted_total": weighted_score, "raw_judgment": judgment}

## Grading Two Customer Support Replies

Let's test the auto-grader on two responses to the same customer complaint — one good, one poor.

In [6]:
task = "Reply to a customer who received a damaged product and wants a replacement."

good_reply = (
    "I'm really sorry to hear your order arrived damaged — that's not the experience "
    "we want for you. I've already initiated a replacement shipment, which should "
    "arrive within 3–5 business days. You don't need to return the damaged item. "
    "I'll send you a tracking number as soon as it ships. Please don't hesitate "
    "to reach out if there's anything else I can help with."
)

poor_reply = (
    "Sorry about that. You can check our returns page for info on how to send it back. "
    "Let us know if you have questions."
)

md(f"**Task:** {task}\n\n"
   f"**Good Reply:** {good_reply}\n\n"
   f"**Poor Reply:** {poor_reply}")

**Task:** Reply to a customer who received a damaged product and wants a replacement.

**Good Reply:** I'm really sorry to hear your order arrived damaged — that's not the experience we want for you. I've already initiated a replacement shipment, which should arrive within 3–5 business days. You don't need to return the damaged item. I'll send you a tracking number as soon as it ships. Please don't hesitate to reach out if there's anything else I can help with.

**Poor Reply:** Sorry about that. You can check our returns page for info on how to send it back. Let us know if you have questions.

In [7]:
grade_good = auto_grade(support_rubric, task, good_reply)

lines = ["### Good Reply — Rubric Scores\n"]
lines.append("| Criterion | Weight | Score | Justification |")
lines.append("|-----------|--------|-------|---------------|")
for name, data in grade_good["scores"].items():
    lines.append(f"| {name} | {data['weight']} | {data['score']}/5 | {data['justification']} |")
lines.append(f"\n**Weighted Total: {grade_good['weighted_total']:.2f} / 5.00**")

md("\n".join(lines))

### Good Reply — Rubric Scores

| Criterion | Weight | Score | Justification |
|-----------|--------|-------|---------------|
| Empathy | 0.3 | 4/5 | The response acknowledges the customer's frustration and expresses regret about the damaged product, which conveys a warm and understanding tone. However, it could be slightly more personal to enhance the feeling of empathy. |
| Resolution | 0.5 | 5/5 | The reply provides a clear and actionable solution by initiating a replacement shipment, specifying the expected delivery timeframe, and stating that the customer does not need to return the damaged item. It also mentions sending a tracking number, which adds to the clarity of the resolution. |
| Professionalism | 0.2 | 5/5 | The response is well-structured, free of grammatical errors, and maintains a professional tone throughout. It aligns well with customer service standards and is appropriate for the situation. |

**Weighted Total: 4.70 / 5.00**

In [8]:
grade_poor = auto_grade(support_rubric, task, poor_reply)

lines = ["### Poor Reply — Rubric Scores\n"]
lines.append("| Criterion | Weight | Score | Justification |")
lines.append("|-----------|--------|-------|---------------|")
for name, data in grade_poor["scores"].items():
    lines.append(f"| {name} | {data['weight']} | {data['score']}/5 | {data['justification']} |")
lines.append(f"\n**Weighted Total: {grade_poor['weighted_total']:.2f} / 5.00**")

md("\n".join(lines))

### Poor Reply — Rubric Scores

| Criterion | Weight | Score | Justification |
|-----------|--------|-------|---------------|
| Empathy | 0.3 | 2/5 | The response offers minimal acknowledgment of the customer's feelings regarding the damaged product. It lacks warmth and does not express understanding or concern for the customer's frustration. |
| Resolution | 0.5 | 3/5 | The reply suggests checking the returns page for information on how to send the product back, which is a solution, but it lacks specific details about the return process or what the customer can expect next. |
| Professionalism | 0.2 | 4/5 | The response is professional and well-structured, with no grammatical errors. However, it could be slightly more polished in tone to enhance the overall customer experience. |

**Weighted Total: 2.90 / 5.00**

## Side-by-Side Comparison

In [9]:
lines = [
    "| Criterion | Weight | Good Reply | Poor Reply |",
    "|-----------|--------|------------|------------|"
]
for c in support_rubric["criteria"]:
    name = c["name"]
    sg = grade_good["scores"].get(name, {}).get("score", "?")
    sp = grade_poor["scores"].get(name, {}).get("score", "?")
    lines.append(f"| {name} | {c['weight']} | {sg}/5 | {sp}/5 |")

lines.append(f"| **Weighted Total** | | **{grade_good['weighted_total']:.2f}/5** | **{grade_poor['weighted_total']:.2f}/5** |")

md("\n".join(lines))

| Criterion | Weight | Good Reply | Poor Reply |
|-----------|--------|------------|------------|
| Empathy | 0.3 | 4/5 | 2/5 |
| Resolution | 0.5 | 5/5 | 3/5 |
| Professionalism | 0.2 | 5/5 | 4/5 |
| **Weighted Total** | | **4.70/5** | **2.90/5** |

Notice how the **weighted total** is dominated by the Resolution criterion (weight 0.5). This is intentional — for customer support, actually solving the problem matters more than tone. Rubric weights let you encode these domain priorities.

## Same Response, Different Rubric

To show why custom rubrics matter, let's evaluate the **same good reply** under a different rubric — one designed for **technical documentation**, where empathy is irrelevant and precision is everything.

In [10]:
docs_rubric = {
    "name": "Technical Documentation",
    "criteria": [
        {
            "name": "Technical Accuracy",
            "description": "Are all technical details correct and precise?",
            "weight": 0.5,
            "levels": {
                1: "Contains factual errors",
                2: "Vague or imprecise technical claims",
                3: "Mostly accurate but lacks precision",
                4: "Accurate with minor gaps",
                5: "Fully accurate and precise"
            }
        },
        {
            "name": "Completeness",
            "description": "Does it cover all necessary parameters, return values, and edge cases?",
            "weight": 0.35,
            "levels": {
                1: "Missing most required information",
                2: "Covers basics but misses key details",
                3: "Adequate coverage with some gaps",
                4: "Thorough with minor omissions",
                5: "Comprehensive — covers all aspects"
            }
        },
        {
            "name": "Code Examples",
            "description": "Does it include working code examples?",
            "weight": 0.15,
            "levels": {
                1: "No code examples",
                2: "Pseudocode or broken examples",
                3: "Basic example that works",
                4: "Good example with comments",
                5: "Multiple examples covering common use cases"
            }
        }
    ]
}

print(f"Rubric: {docs_rubric['name']} — {len(docs_rubric['criteria'])} criteria")

Rubric: Technical Documentation — 3 criteria


In [11]:
# Grade the same good support reply under the docs rubric
grade_wrong_rubric = auto_grade(
    docs_rubric,
    "Write technical documentation for an API endpoint.",
    good_reply
)

lines = ["### Good Support Reply — Scored by Technical Documentation Rubric\n"]
lines.append("| Criterion | Weight | Score | Justification |")
lines.append("|-----------|--------|-------|---------------|")
for name, data in grade_wrong_rubric["scores"].items():
    lines.append(f"| {name} | {data['weight']} | {data['score']}/5 | {data['justification']} |")
lines.append(f"\n**Weighted Total: {grade_wrong_rubric['weighted_total']:.2f} / 5.00**")
lines.append(f"\n(Compared to **{grade_good['weighted_total']:.2f}/5** under the Support rubric)")

md("\n".join(lines))

### Good Support Reply — Scored by Technical Documentation Rubric

| Criterion | Weight | Score | Justification |
|-----------|--------|-------|---------------|
| Technical Accuracy | 0.5 | 1/5 | The response does not contain any technical details related to an API endpoint, as it is a customer service message regarding a damaged order. |
| Completeness | 0.35 | 1/5 | The response lacks any information about parameters, return values, or edge cases relevant to an API endpoint. It does not address any technical aspects that would be necessary for documentation. |
| Code Examples | 0.15 | 1/5 | There are no code examples provided in the response, as it is not related to API documentation at all. |

**Weighted Total: 1.00 / 5.00**

(Compared to **4.70/5** under the Support rubric)

The same response that scored high under the customer support rubric scores low under the technical documentation rubric. **The rubric defines what "good" means** — and that definition changes with the domain.

This is exactly why generic evaluation criteria aren't enough. Custom rubrics ensure your auto-grader measures what actually matters for your specific use case.

## Rubric Design Tips

| Tip | Why |
|-----|-----|
| **3–5 criteria** | More than 5 dilutes focus and makes scoring noisy |
| **Weights must sum to 1.0** | Makes the weighted total interpretable on the same 1–5 scale |
| **Define every score level** | Without level definitions, the judge interprets scores inconsistently |
| **Weight by business impact** | The criterion that matters most to your users should have the highest weight |
| **Test with known good/bad examples** | Validate that the rubric produces expected score spreads before using it at scale |

## 🎯 Summary

### Key Takeaways

1. **Rubric scoring formalizes "good"** — by defining criteria, score levels, and weights, you make evaluation explicit and reproducible instead of relying on vague judgments
2. **Weights encode domain priorities** — a customer support rubric weights resolution highest, while a documentation rubric weights accuracy; the same response scores very differently under each
3. **Score-level definitions are critical** — telling the judge what a 3 vs. a 5 means for each criterion produces consistent, interpretable scores
4. **The auto-grader pattern is reusable** — define a rubric dict, convert it to a prompt, call the LLM judge, parse and weight the scores; this pattern works for any domain